# Kazakh Fact-Checking Benchmark — прогон Gemini в Google Colab

Из пяти моделей по API гоняется только **Gemini** (остальные — через веб-чат). Colab даёт открытый интернет и хранит ключ в Secrets.

## Порядок действий
1. Слева 🔑 (**Secrets**) → добавь `GEMINI_API_KEY` (ключ Google AI Studio), включи **Notebook access**.
2. В ячейке «Настройки» выбери `SOURCE`.
3. Выполняй ячейки сверху вниз (Shift+Enter).

⚠️ Ключ храни ТОЛЬКО в Secrets. Не вставляй его в ячейки и не коммить.

### 1. Клонируем репозиторий и ставим зависимости

In [ ]:
%cd /content
BRANCH = 'claude/kazakh-factcheck-benchmark-e76kd5'
REPO = 'github.com/Tim2190/kazakh-factcheck-benchmark.git'
!rm -rf /content/kazakh-factcheck-benchmark
!git clone --branch $BRANCH https://$REPO
%cd /content/kazakh-factcheck-benchmark
!pip install -q -r requirements.txt
print('OK')

### 2. Настройки: какой текст гоняем
`leg_text01` — Конституция (~50k токенов). `news_text1` — новость (~5k). Добавятся новые — просто поменяй имя.

In [ ]:
SOURCE = 'news_text1'

import os
from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
print('GEMINI_API_KEY загружен')

### 3. Прогон Gemini (батч) + проверка заземления + скоринг
`--batch` — весь документ + все тезисы одним запросом. Модель по умолчанию `gemini-2.5-flash`; при `429 / limit: 0` поменяй `model_id` в `scripts/models.json` (напр. `gemini-3.1-flash-lite`).

In [ ]:
!python scripts/export_dataset.py
!python scripts/run_factcheck.py --model gemini --batch --source $SOURCE
!python scripts/check_grounding.py results/gemini_{SOURCE}_run.json
!python scripts/score.py results/gemini_{SOURCE}_run.json

### 4. Скачать результат и прислать мне

In [ ]:
from google.colab import files
files.download(f'results/gemini_{SOURCE}_run.json')